# অলীকবচন Phase 2: offline hallucination-detection pipeline

Single self-contained notebook, no internet at inference, per Phase-2 rules. Architecture is
the **v19** design (validated Phase-1 LB 0.739, wcv 0.7777 -- the best *and* fastest
candidate found this session, replacing the heavier v17 stack that relied on ~950
non-reproducible manually-judged rows):

1. **NLI lane** -- `MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7` (SummaC-ZS
   style entailment/contradiction over context windows), packaged as Kaggle Dataset
   `alveeahnaf/mdeberta-v3-xnli-multilingual` (no matching official Kaggle Model exists).
2. **Judge lane** -- `google/gemma-4-12b-it`, the *official* Google checkpoint via the
   **Kaggle Model** `google/gemma-4/transformers/gemma-4-12b-it` (zero data transfer,
   Apache 2.0), two-polarity Yes/No next-token-logit scoring (Bengali + English prompts).
   This REPLACES the earlier Qwen3-8B + gemma-3 cross-lingual stack entirely -- gemma-4
   alone scored the best wcv of the whole session (0.7777) at ~1/4 the compute.
3. **Math override** -- `math_solver.py`'s deterministic template solver for ~15 Bengali
   word-problem types (work-rate, ratio, profit-loss, simple interest, etc.), 100%
   reproducible, no model needed.
4. **Decision layer** -- GaussianNB over {llm, pA, pB, |pA-pB|, nli, nli_soft, has_ctx,
   resp_chars, ctx_chars, math_override}, trained fresh on the labeled sample using
   features computed by THIS notebook (not reused from any earlier kernel -- Phase 1
   predictions must be reproducible by the Phase 2 package, so nothing here may depend on
   a human-in-the-loop judgment step).

Offline plumbing (wheelhouse-based `transformers` upgrade, both model sources resolving
with `enable_internet: false`) was independently validated end-to-end in the
`phase2-offline-test` kernel before this notebook was written.

In [1]:
# ---------- bootstrap: upgrade transformers from a local wheelhouse (NO internet needed/used) ----------
# gemma4_unified is only recognized by transformers>=5.13; Kaggle's frozen image ships 5.0.0.
# Must run before anything else imports transformers, so the upgrade is picked up cleanly.
import subprocess, sys, glob, os

hits = glob.glob('/kaggle/input/**/wheelhouse', recursive=True)
assert hits, ('offline wheelhouse not found under /kaggle/input -- attach the wheelhouse-builder '
              'kernel output via kernel_sources')
WHEELDIR = hits[0]
print('wheelhouse found at:', WHEELDIR, '|', len(glob.glob(f"{WHEELDIR}/*.whl")), 'wheels')

r = subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--find-links', WHEELDIR,
                   '-U', 'transformers', 'accelerate', 'bitsandbytes'],
                   capture_output=True, text=True, timeout=300)
print(r.stdout[-2000:])
if r.returncode != 0:
    print('STDERR:', r.stderr[-2000:])
assert r.returncode == 0, 'offline install from wheelhouse failed'

import transformers
print('transformers version:', transformers.__version__)

wheelhouse found at: /kaggle/input/notebooks/alveeahnaf/wheelhouse-builder/wheelhouse | 7 wheels
ttpx<1,>=0.23.0->huggingface-hub<2.0,>=1.5.0->transformers) (2026.4.22)
  Attempting uninstall: safetensors
    Found existing installation: safetensors 0.7.0
    Uninstalling safetensors-0.7.0:
      Successfully uninstalled safetensors-0.7.0
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.13.0
    Uninstalling accelerate-1.13.0:
      Successfully uninstalled accelerate-1.13.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0

transformers version: 5.13.0


In [2]:
import os
# fight fragmentation (must be set before the first CUDA allocation)
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')
import re, glob, json, random, gc, warnings
import numpy as np, pandas as pd
import torch
warnings.filterwarnings('ignore')

nli_hits = [p for p in glob.glob('/kaggle/input/**/config.json', recursive=True) if 'mdeberta' in p.lower()]
assert nli_hits, 'mDeBERTa dataset not found under /kaggle/input -- check dataset_sources'
NLI_PATH = os.path.dirname(nli_hits[0])
print('NLI model resolved at:', NLI_PATH)

CFG = {
    # ---------- paths ----------
    'comp_dir'  : None,    # None -> auto-scan /kaggle/input/bengali-hallucination, /kaggle/input, ./data
    'test_csv'  : None,
    'labeled_csv': None,   # discovery also reads .json (comp ships 'dataset samples.json')
    'nli_path'  : NLI_PATH,
    # ---------- column overrides (None = auto-detect) ----------
    'col_id': None, 'col_prompt': None, 'col_response': None, 'col_context': None, 'col_label': None,
    # ---------- inference ----------
    'llm_batch' : 8,
    'llm_token_budget': 12288,
    'nli_batch' : 64,
    'win_list'  : [2, 3],
    'max_ctx_chars': 3000, 'max_prompt_chars': 1500, 'max_resp_chars': 2500,
    'max_len_llm'  : 2048,
    'seed': 42,
}

random.seed(CFG['seed']); np.random.seed(CFG['seed']); torch.manual_seed(CFG['seed'])
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(DEVICE, '|', torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'no GPU — enable an accelerator',
      '| GPUs:', torch.cuda.device_count() if DEVICE == 'cuda' else 0)

NLI model resolved at: /kaggle/input/datasets/alveeahnaf/mdeberta-v3-xnli-multilingual
cuda | Tesla T4 | GPUs: 2


In [3]:
# ---------- file discovery (CSV + JSON) + column auto-detection ----------
CAND = {
    'id'      : ['id', 'ID', 'Id', 'idx', 'index', 'sample_id'],
    'prompt'  : ['prompt', 'question', 'query', 'instruction', 'input'],
    'response': ['response', 'answer', 'candidate', 'output', 'model_response',
                 'generated_response', 'candidate_response', 'hypothesis'],
    'context' : ['context', 'passage', 'source', 'source_passage', 'evidence',
                 'document', 'reference', 'source_text'],
    'label'   : ['label', 'target', 'TARGET', 'y', 'gold', 'is_faithful'],
}

def autodetect(cols, key):
    if CFG.get('col_' + key):
        return CFG['col_' + key]
    low = {c.lower(): c for c in cols}
    for cand in CAND[key]:
        if cand.lower() in low:
            return low[cand.lower()]
    for c in cols:
        if key in c.lower():
            return c
    return None

def resolve(df, keys):
    return {k: autodetect(list(df.columns), k) for k in keys}

def role_of(df):
    cols = list(df.columns)
    has = {k: autodetect(cols, k) is not None for k in CAND}
    if has['id'] and has['label'] and len(cols) <= 2:
        return 'sample_submission'
    if has['response'] and has['label']:
        return 'labeled'
    if has['response']:
        return 'test'
    return 'other'

def read_any(p):
    if p.lower().endswith('.csv'):
        return pd.read_csv(p)
    with open(p, encoding='utf-8') as f:
        obj = json.load(f)
    if isinstance(obj, list) and obj and isinstance(obj[0], dict):
        return pd.DataFrame(obj)
    raise ValueError('not a list-of-records json')

ROOTS = [r for r in ([CFG['comp_dir']] if CFG['comp_dir'] else
                     ['/kaggle/input/bengali-hallucination',
                      '/kaggle/input/competitions/bengali-hallucination',
                      '/kaggle/input', 'data']) if r and os.path.isdir(r)]
assert ROOTS, 'No input roots found — set CFG["comp_dir"].'

seen, files = set(), []
for root in ROOTS:
    for p in sorted(glob.glob(os.path.join(root, '**', '*.csv'), recursive=True)
                    + glob.glob(os.path.join(root, '**', '*.json'), recursive=True)):
        rp = os.path.realpath(p)
        if rp in seen or os.path.getsize(p) > 80_000_000:
            continue
        seen.add(rp); files.append(p)

test_df, labeled_df = None, None
for p in files:
    try:
        df = read_any(p)
    except Exception as e:
        if p.lower().endswith('.csv'):   # json misses are usually model configs — stay quiet
            print('skip', os.path.basename(p), '—', e)
        continue
    r = role_of(df)
    if r != 'other':
        print(f"{os.path.basename(p):40s} shape={str(df.shape):14s} -> {r}")
    if r == 'test'    and (test_df    is None or len(df) > len(test_df)):    test_df    = df.copy()
    if r == 'labeled' and (labeled_df is None or len(df) > len(labeled_df)): labeled_df = df.copy()

if CFG['test_csv']:    test_df    = read_any(CFG['test_csv'])
if CFG['labeled_csv']: labeled_df = read_any(CFG['labeled_csv'])
assert test_df is not None, 'Could not identify the test file — set CFG["test_csv"] explicitly.'

COLS  = resolve(test_df, ['id', 'prompt', 'response', 'context'])
LCOLS = resolve(labeled_df, ['id', 'prompt', 'response', 'context', 'label']) if labeled_df is not None else None
print('\nResolved test columns:', COLS)
if LCOLS: print('Resolved labeled columns:', LCOLS)
print('test:', test_df.shape, '| labeled sample:', None if labeled_df is None else labeled_df.shape)

dataset samples.json                     shape=(299, 4)       -> labeled
sample submission.csv                    shape=(2516, 2)      -> sample_submission
test set.csv                             shape=(2516, 4)      -> test
dataset samples.json                     shape=(299, 4)       -> labeled

Resolved test columns: {'id': 'id', 'prompt': 'prompt_bn', 'response': 'response_bn', 'context': 'context'}
Resolved labeled columns: {'id': None, 'prompt': 'prompt_bn', 'response': 'response_bn', 'context': 'context', 'label': 'label'}
test: (2516, 4) | labeled sample: (299, 4)


In [4]:
# ---------- prep ----------
NULLISH = {'', '[null]', 'null', 'none', 'nan', 'n/a'}   # the comp files use the literal string '[NULL]'

def clean(s, lim):
    if pd.isna(s):
        return ''
    s = str(s).strip()
    return '' if s.lower() in NULLISH else s[:lim]

def prep(df, cols):
    d = pd.DataFrame()
    d['id']       = df[cols['id']].values if cols['id'] else np.arange(len(df))
    d['prompt']   = df[cols['prompt']].map(lambda s: clean(s, CFG['max_prompt_chars'])).values if cols['prompt'] else ''
    d['response'] = df[cols['response']].map(lambda s: clean(s, CFG['max_resp_chars'])).values
    d['context']  = df[cols['context']].map(lambda s: clean(s, CFG['max_ctx_chars'])).values if cols['context'] else ''
    d['has_ctx']  = d['context'].str.len() > 0
    return d

test  = prep(test_df, COLS)
train = None
if labeled_df is not None and LCOLS and LCOLS['label']:
    train = prep(labeled_df, LCOLS)
    train['y'] = labeled_df[LCOLS['label']].astype(int).values   # 1 = faithful, 0 = hallucinated
    assert set(np.unique(train['y'])) <= {0, 1}, 'unexpected label values'
    print('sample label balance (1=faithful):')
    print(train['y'].value_counts(normalize=True).round(3).to_dict())
print('has_context — test:', round(test['has_ctx'].mean(), 3),
      '| sample:', None if train is None else round(train['has_ctx'].mean(), 3))

BN_SENT = re.compile(r'(?<=[।!?\n])\s+')   # Bengali danda-aware sentence split
def sents(t):
    parts = [s.strip() for s in BN_SENT.split(t) if s.strip()]
    return parts if parts else ([t] if t else [])

sample label balance (1=faithful):
{1: 0.545, 0: 0.455}
has_context — test: 0.541 | sample: 0.435


In [5]:
# ---------- Lane A: zero-shot multilingual NLI over context windows ----------
from transformers import AutoTokenizer, AutoModelForSequenceClassification

nli_id  = CFG['nli_path'] or 'MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7'
nli_tok = AutoTokenizer.from_pretrained(nli_id)
try:
    nli_mod = AutoModelForSequenceClassification.from_pretrained(nli_id, dtype=torch.float16)
except TypeError:
    nli_mod = AutoModelForSequenceClassification.from_pretrained(nli_id, torch_dtype=torch.float16)
nli_mod = nli_mod.to(DEVICE).eval()
lab2id  = {v.lower(): k for k, v in nli_mod.config.id2label.items()}
E_ID, C_ID = lab2id['entailment'], lab2id['contradiction']
print('NLI labels:', nli_mod.config.id2label)

@torch.no_grad()
def nli_batch(premises, hyps):
    out = []
    for i in range(0, len(premises), CFG['nli_batch']):
        enc = nli_tok(premises[i:i + CFG['nli_batch']], hyps[i:i + CFG['nli_batch']],
                      truncation=True, max_length=512, padding=True, return_tensors='pt').to(DEVICE)
        out.append(torch.softmax(nli_mod(**enc).logits.float(), -1).cpu().numpy())
    return np.concatenate(out)

def ctx_windows(ctx, win, cap=16):
    ss = sents(ctx)
    if len(ss) <= win:
        return [' '.join(ss)] if ss else []
    stride = max(1, win - 1)
    wins = [' '.join(ss[i:i + win]) for i in range(0, len(ss) - win + 1, stride)]
    wins.append(' '.join(ss[-win:]))
    return wins[:cap]

def nli_score_rows(df, tag=''):
    """SummaC-ZS style, ensembled over CFG['win_list'] window sizes (roadmap #1).
    strict: min over claims of max-entailment  -  max over claims of max-contradiction
    soft  : mean over claims of (max-entailment - max-contradiction)     both in [-1, 1]"""
    strict = np.full(len(df), np.nan, dtype=float)
    soft   = np.full(len(df), np.nan, dtype=float)
    pos = np.where(df['has_ctx'].values)[0]
    for j, i in enumerate(pos):
        ctx, resp = df['context'].iat[i], df['response'].iat[i]
        hyp = sents(resp)[:12] or [resp]
        st, so = [], []
        for win in CFG['win_list']:
            wins = ctx_windows(ctx, win) or [ctx]
            P, H = [], []
            for h in hyp:
                for w in wins:
                    P.append(w); H.append(h)
            probs = nli_batch(P, H).reshape(len(hyp), len(wins), -1)
            ent = probs[:, :, E_ID].max(axis=1)
            con = probs[:, :, C_ID].max(axis=1)
            st.append(float(ent.min() - con.max()))
            so.append(float((ent - con).mean()))
        strict[i] = float(np.mean(st)); soft[i] = float(np.mean(so))
        if (j + 1) % 200 == 0:
            print(f'  nli[{tag}] {j + 1}/{len(pos)}')
    return strict, soft

# run Lane A for both frames NOW, then give the GPU back before the judge loads
nli_tr = nli_score_rows(train, 'sample') if train is not None else None
nli_te = nli_score_rows(test,  'test')
nli_mod = nli_mod.to('cpu'); gc.collect(); torch.cuda.empty_cache()
print('NLI lane done — model offloaded to CPU')

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

NLI labels: {0: 'entailment', 1: 'neutral', 2: 'contradiction'}
  nli[test] 200/1361
  nli[test] 400/1361
  nli[test] 600/1361
  nli[test] 800/1361
  nli[test] 1000/1361
  nli[test] 1200/1361
NLI lane done — model offloaded to CPU


In [6]:
# ---------- deterministic math-word-problem solver (100% reproducible, no model) ----------
BN2EN = str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789")
DAYS = ["রবিবার", "সোমবার", "মঙ্গলবার", "বুধবার", "বৃহস্পতিবার", "শুক্রবার", "শনিবার"]


def _nums(s):
    s = re.sub(r"(?<=[০-৯]),(?=[০-৯])", "", s)
    return [int(x.translate(BN2EN)) for x in re.findall(r"[০-৯]+", s)]


def _given_num(resp):
    resp = re.sub(r"(?<=[০-৯]),(?=[০-৯])", "", resp)
    m = re.findall(r"[০-৯]+", resp)
    return int(m[0].translate(BN2EN)) if m else None


def _close(a, b, tol=0.5):
    return a is not None and b is not None and abs(a - b) < tol


def solve_row(prompt, response):
    """Returns 1 (faithful) / 0 (hallucinated) / None (no template matched)."""
    n = _nums(prompt)
    g = _given_num(response)

    if "সপ্তাহের কোন দিন" in prompt or "সপ্তাহের কোন বার" in prompt:
        day_names = [d for d in DAYS if d in prompt]
        if not day_names or not n:
            return None
        idx = (DAYS.index(day_names[0]) + n[0]) % 7
        return int(DAYS[idx] in response)

    if len(n) < 2:
        return None

    if re.search("একা একটি কাজ|একাই.*দিনে সমাপ্ত", prompt):
        a, b = n[0], n[1]
        correct = (a * b) / (a + b) if (a + b) else 0
        return int(_close(correct, g))

    if "ক, খ ও গ" in prompt:
        a, b, c = n[0], n[1], n[2]
        rate = (1 / a if a else 0) + (1 / b if b else 0) + (1 / c if c else 0)
        correct = 1 / rate if rate else 0
        return int(_close(correct, g))

    if "বয়সের অনুপাত" in prompt:
        a, b, total = n[0], n[1], n[2]
        share = min(a, b)
        correct = total * share / (a + b)
        return int(_close(correct, g))

    if "সংখ্যার অনুপাত" in prompt:
        a, b, total = n[0], n[1], n[2]
        correct = total * b / (a + b)
        return int(_close(correct, g))

    if "ক্রয়মূল্য" in prompt or "কেনা হয়েছিল" in prompt:
        cost, pct = n[0], n[1]
        correct = cost * (1 - pct / 100) if "ক্ষতি" in prompt else cost * (1 + pct / 100)
        return int(_close(correct, g))

    if "প্রথম দফায়" in prompt or "প্রাথমিক মূল্য" in prompt:
        start_m = re.search(r"(?:প্রাথমিক মূল্য|শুরুর দাম)\s*([০-৯,]+)", prompt)
        if not start_m:
            return None
        start = int(re.sub(",", "", start_m.group(1)).translate(BN2EN))
        pcts = _nums(re.sub(r"(?:প্রাথমিক মূল্য|শুরুর দাম)\s*[০-৯,]+", "", prompt))
        p1, p2 = pcts[0], pcts[1]
        seg1 = prompt.split("এরপর" if "এরপর" in prompt else "পরে", 1)[0]
        seg2 = prompt.split("এরপর", 1)[-1] if "এরপর" in prompt else prompt.split("পরে", 1)[-1]
        s1 = 1 + p1 / 100 if ("বৃদ্ধি" in seg1 or "বেড়ে" in seg1) else 1 - p1 / 100
        s2 = 1 - p2 / 100 if ("কমে" in seg2 or "ছাড়" in seg2) else 1 + p2 / 100
        correct = start * s1 * s2
        return int(_close(correct, g))

    if "সরল সুদ" in prompt:
        principal, rate, time = n[0], n[1], n[2]
        correct = principal * rate * time / 100
        return int(_close(correct, g))

    if "অংশীদারের মধ্যে" in prompt:
        total, a, b, c = n[0], n[1], n[2], n[3]
        correct = total * b / (a + b + c)
        return int(_close(correct, g))

    if "একই দিকে দুইটি বাস" in prompt or "সাইকেল আরোহী একই বিন্দু" in prompt:
        s1, s2, t = n[0], n[1], n[2]
        correct = abs(s1 - s2) * t
        return int(_close(correct, g))

    if "দুইটি শহরের মধ্যে দূরত্ব" in prompt:
        dist, s1, s2 = n[0], n[1], n[2]
        correct = dist / (s1 + s2) if (s1 + s2) else 0
        return int(_close(correct, g))

    if "সংকেত বাতি" in prompt or "বাস স্টপেজ থেকে বাস" in prompt:
        from math import gcd
        a, b, c = n[0], n[1], n[2]
        lcm2 = a * b // gcd(a, b)
        correct = lcm2 * c // gcd(lcm2, c)
        return int(_close(correct, g))

    if "মিশ্রণে চিনি" in prompt:
        a, b, total = n[0], n[1], n[2]
        correct = total * b / (a + b)
        return int(_close(correct, g))

    if "প্যানেল গঠন" in prompt or "উপকমিটি গঠন" in prompt:
        from math import comb
        total, r = n[0], n[1]
        correct = comb(total, r)
        return int(_close(correct, g))

    if "রাশির গড়মান" in prompt or "শিক্ষার্থীর গড় নম্বর" in prompt:
        count, avg1, avg2 = n[0], n[1], n[2]
        correct = avg2 * (count + 1) - avg1 * count
        return int(_close(correct, g))

    return None


def math_feature(df):
    out = np.full(len(df), np.nan)
    for i in range(len(df)):
        lbl = solve_row(df['prompt'].iat[i], df['response'].iat[i])
        if lbl is not None:
            out[i] = lbl
    return out


math_tr = math_feature(train) if train is not None else None
math_te = math_feature(test)
print('math_solver matched -- sample:', None if math_tr is None else int(np.sum(~np.isnan(math_tr))), '/',
      0 if math_tr is None else len(math_tr),
      '| test:', int(np.sum(~np.isnan(math_te))), '/', len(math_te))

math_solver matched -- sample: 0 / 299 | test: 136 / 2516


In [7]:
# ---------- judge lane: gemma-4-12b-it (official Google checkpoint, offline via Kaggle Model) ----------
import kagglehub
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

GEMMA_PATH = kagglehub.model_download('google/gemma-4/transformers/gemma-4-12b-it')
print('gemma-4-12b-it resolved at:', GEMMA_PATH)

tok = AutoTokenizer.from_pretrained(GEMMA_PATH)
tok.padding_side = 'left'; tok.truncation_side = 'left'
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

qcfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                          bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float32)
kw = dict(device_map='auto', attn_implementation='eager', quantization_config=qcfg)
try:
    llm = AutoModelForCausalLM.from_pretrained(GEMMA_PATH, dtype=torch.float16, **kw).eval()
except TypeError:
    llm = AutoModelForCausalLM.from_pretrained(GEMMA_PATH, torch_dtype=torch.float16, **kw).eval()
print('loaded gemma-4-12b-it, 4-bit, eager attention')


def token_set(variants):
    ids = []
    for v in variants:
        e = tok.encode(v, add_special_tokens=False)
        if len(e) == 1 and e[0] not in ids:
            ids.append(e[0])
    return ids


YES = token_set(['Yes', 'yes', ' Yes', ' yes', 'হ্যাঁ'])
NO = token_set(['No', 'no', ' No', ' no', 'না'])
assert YES and NO and not set(YES) & set(NO)


def chat(user_text):
    msgs = [{'role': 'user', 'content': user_text}]
    try:
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True,
                                       enable_thinking=False)
    except TypeError:
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)


def prompt_A(p, r, c):
    ctx = f"\n\nতথ্যসূত্র (এটিই একমাত্র সত্যের ভিত্তি):\n{c}" if c else ""
    src = "প্রদত্ত তথ্যসূত্র" if c else "বাস্তব, যাচাইযোগ্য জ্ঞান"
    return chat(f"""তুমি একজন কঠোর বাংলা ফ্যাক্ট-চেকার।{ctx}

প্রশ্ন/নির্দেশ:
{p}

মডেলের উত্তর:
{r}

উপরের উত্তরটি কি {src}-এর সাথে সম্পূর্ণ সঙ্গতিপূর্ণ — কোনো ভুল তথ্য, বানোয়াট নাম/তারিখ/সংখ্যা বা অসমর্থিত দাবি নেই? শুধু এক শব্দে উত্তর দাও: Yes অথবা No।""")


def prompt_B(p, r, c):
    ctx = f"\n\nSOURCE (the only ground truth):\n{c}" if c else ""
    tail = "not supported by the SOURCE" if c else "false in the real world"
    return chat(f"""You are a strict fact-checker for Bengali LLM outputs.{ctx}

PROMPT:
{p}

RESPONSE:
{r}

Does the RESPONSE contain any hallucination — a factual error, a fabricated name/date/number, or a claim {tail}? Answer with exactly one word: Yes or No.""")


@torch.no_grad()
def judge(df, builder, flip, tag=''):
    texts = [builder(df['prompt'].iat[i], df['response'].iat[i], df['context'].iat[i])
             for i in range(len(df))]
    lens = np.array([len(x) for x in tok(texts, truncation=True,
                                         max_length=CFG['max_len_llm'])['input_ids']])
    order = np.argsort(lens, kind='stable')
    pair_ids = YES + NO
    ps, i, done = np.zeros(len(texts)), 0, 0
    while i < len(order):
        take = int(min(CFG['llm_batch'], len(order) - i))
        while take > 1 and int(lens[order[i + take - 1]]) * take > CFG['llm_token_budget']:
            take -= 1
        idx = order[i:i + take]
        while True:
            try:
                enc = tok([texts[k] for k in idx], return_tensors='pt', padding=True,
                          truncation=True, max_length=CFG['max_len_llm']).to(llm.device)
                try:
                    out = llm(**enc, use_cache=False, logits_to_keep=1)
                except TypeError:
                    out = llm(**enc, use_cache=False)
                break
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                if len(idx) == 1:
                    raise
                idx = idx[:max(1, len(idx) // 2)]
                print(f'  judge[{tag}] OOM -> retry with batch {len(idx)}')
        logits = out.logits[:, -1, :].float()
        soft = torch.softmax(logits[:, pair_ids], -1).cpu().numpy()
        ps[idx] = soft[:, :len(YES)].sum(axis=1)
        i += len(idx); done += len(idx)
        if done // 300 != (done - len(idx)) // 300:
            print(f'  judge[{tag}] {done}/{len(texts)}')
    return 1.0 - ps if flip else ps


pA_tr = judge(train, prompt_A, flip=False, tag='sample/A') if train is not None else None
pB_tr = judge(train, prompt_B, flip=True, tag='sample/B') if train is not None else None
pA_te = judge(test, prompt_A, flip=False, tag='test/A')
pB_te = judge(test, prompt_B, flip=True, tag='test/B')

del llm, tok
gc.collect(); torch.cuda.empty_cache()
print('judge lane done')

gemma-4-12b-it resolved at: /kaggle/input/models/google/gemma-4/transformers/gemma-4-12b-it/2


Loading weights:   0%|          | 0/677 [00:00<?, ?it/s]

loaded gemma-4-12b-it, 4-bit, eager attention
  judge[test/A] 304/2516
  judge[test/A] 600/2516
  judge[test/A] 904/2516
  judge[test/A] 1200/2516
  judge[test/A] 1504/2516
  judge[test/A] 1800/2516
  judge[test/A] 2104/2516
  judge[test/A] 2400/2516
  judge[test/B] 304/2516
  judge[test/B] 600/2516
  judge[test/B] 904/2516
  judge[test/B] 1200/2516
  judge[test/B] 1504/2516
  judge[test/B] 1800/2516
  judge[test/B] 2104/2516
  judge[test/B] 2400/2516
judge lane done


In [8]:
# ---------- assemble features: gemma-4 IS the base judge (v19 architecture, no Qwen3-8B / gemma-3-pT) ----------
def assemble(df, pA, pB, nli_pair, math_arr):
    s_nli, s_soft = nli_pair
    llm = (pA + pB) / 2
    return pd.DataFrame({
        'llm': llm, 'pA': pA, 'pB': pB,
        'nli': s_nli, 'nli_soft': s_soft,
        'has_ctx': df['has_ctx'].values,
        'resp_chars': df['response'].str.len().values,
        'ctx_chars': df['context'].str.len().values,
        'math_override': math_arr,
    })


feat_tr = assemble(train, pA_tr, pB_tr, nli_tr, math_tr) if train is not None else None
feat_te = assemble(test, pA_te, pB_te, nli_te, math_te)
assert feat_tr is not None, 'labeled sample ("dataset samples.json") not found -- cannot fit the decision layer'


def X_of(feat):
    cols = [feat['llm'].values, feat['pA'].values, feat['pB'].values,
            np.abs(feat['pA'].values - feat['pB'].values),
            (np.nan_to_num(feat['nli'].values, nan=0.0) + 1) / 2,
            (np.nan_to_num(feat['nli_soft'].values, nan=0.0) + 1) / 2,
            feat['has_ctx'].values.astype(float),
            np.minimum(feat['resp_chars'].values, CFG['max_resp_chars']) / CFG['max_resp_chars'],
            np.minimum(feat['ctx_chars'].values, CFG['max_ctx_chars']) / CFG['max_ctx_chars'],
            np.nan_to_num(feat['math_override'].values.astype(float), nan=0.5)]
    return np.column_stack(cols)


feat_tr.assign(y=train['y'].values).to_csv('sample_features.csv', index=False)
feat_te.to_csv('test_features.csv', index=False)
print('features assembled -- sample:', feat_tr.shape, '| test:', feat_te.shape)

features assembled -- sample: (299, 9) | test: (2516, 9)


In [9]:
# ---------- decision layer: GaussianNB, trained fresh on this notebook's own features ----------
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import f1_score, classification_report

y = train['y'].values
X, Xte = X_of(feat_tr), X_of(feat_te)

gnb = GaussianNB().fit(X, y)
oof_probs = gnb.predict_proba(X)[:, 1]


def tune_thr(y, probs):
    best = (0.5, -1.0)
    for t in np.linspace(0.05, 0.95, 91):
        f1 = f1_score(y, (probs >= t).astype(int), average='macro')
        if f1 > best[1]:
            best = (float(t), float(f1))
    return best


thr, f1_in_sample = tune_thr(y, oof_probs)
print(f'in-sample GNB thr={thr:.3f} macro-F1={f1_in_sample:.4f}')
print(classification_report(y, (oof_probs >= thr).astype(int),
                            target_names=['hallucinated(0)', 'faithful(1)'], digits=3))

pte = gnb.predict_proba(Xte)[:, 1]
preds = (pte >= thr).astype(int)
sub = pd.DataFrame({'id': test['id'].values, 'label': preds})
sub.to_csv('submission.csv', index=False)
print('\nwrote submission.csv | balance (1=faithful):',
      sub['label'].value_counts(normalize=True).round(3).to_dict())
sub.head()

in-sample GNB thr=0.180 macro-F1=0.7774
                 precision    recall  f1-score   support

hallucinated(0)      0.757     0.757     0.757       136
    faithful(1)      0.798     0.798     0.798       163

       accuracy                          0.779       299
      macro avg      0.777     0.777     0.777       299
   weighted avg      0.779     0.779     0.779       299


wrote submission.csv | balance (1=faithful): {1: 0.516, 0: 0.484}


,id,label
0,1,1
1,2,1
2,3,1
3,4,1
4,5,1
